# VisDrone — yolo26**m**

Lam tron mot size: baseline -> prune 50% -> finetune + CWD -> val.
Ket qua la **hai dong** cua bang: `YOLO26-M` va `Ours-M`.

| | |
|---|---|
| Baseline | `yolo26m.pt` (COCO), 100 epoch tren VisDrone |
| Ours | L1-norm uniform 50% (div 8) + 100 epoch CWD tau=9, kd_layers=neck |
| Batch / imgsz / seed | 16 / 640 / 0 |
| cos_lr / patience / warmup | False / 100 / 3.0 |
| Uoc tinh | ~15h, **2 phien** |

> Bon notebook n/s/m/l dung **y het** cac tham so nay. Doi mot cai thoi la
> ca bang het so sanh duoc. Dung sua `EPOCHS`, `BATCH`, `IMGSZ`, `RATIO`.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
   (can Internet: VisDrone 2.3 GB va `yolo26m.pt` deu tai tu mang)
2. Bam **Save & Run All**. Lan dau khong can Add Data.
3. Phien tu dung o 10h. Cell cuoi bao con thieu bao nhieu epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai. Lap den khi bao `XONG`.
4. Xong thi gui lai bang 2 dong o cell cuoi.

Notebook chay tuan tu baseline roi moi den Ours. Neu het gio giua chung, cac
cell sau tu bo qua va in ra con thieu gi — khong bao loi.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [ ]:
import os, sys, glob, shutil, pathlib, subprocess

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "pruning"))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
from ultralytics import YOLO
print("GPU:", torch.cuda.device_count())

## 2. Cau hinh

In [ ]:
SIZE   = "m"
MODEL  = "yolo26m.pt"
RATIO  = 0.5

DATA   = "VisDrone.yaml"   # Ultralytics tu tai 2.3 GB, can Internet: On
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640
DEVICE = "0,1" if torch.cuda.device_count() > 1 else "0"
STOP_AFTER_H = 10.0        # tu dung truoc moc 12h de output kip luu

# Chot cung ba tham so nay thay vi de mac dinh, vi cac run VOC truoc day KHONG
# dong nhat: n/s/l chay batch 32 + cos_lr=True + patience 20-30, con m chay
# batch 16 + cos_lr=False + patience 100. Bang theo size ma moi size mot config
# thi khong so sanh duoc. Lay config cua m vi do la cau hinh chinh cua bai.
# patience=100 = tat early stop -> ca 4 size deu la run 100 epoch that.
COS_LR   = False
PATIENCE = 100
WARMUP   = 3.0

BASE_NAME = "vd_yolo26m"
OURS_NAME = "vd_oursm"
PRUNED = REPO_DIR / "weights" / "yolo26m_vd_pruned50.pt"

print(BASE_NAME, "|", OURS_NAME, "|", DEVICE)

## 3. Ham dung chung

In [ ]:
# Hai ham nho dung chung cho ca hai lan train duoi day.

def done_epochs(name):
    """So epoch da train xong, doc tu results.csv."""
    csv = REPO_DIR / "runs" / name / "results.csv"
    if not csv.exists():
        return 0
    rows = [r for r in csv.read_text().strip().splitlines()[1:] if r.strip()]
    return int(float(rows[-1].split(",")[0])) if rows else 0


def train(name, weights, **extra):
    """Train moi, hoac train TIEP neu phien truoc bi cat ngang.

    Co last.pt la resume. Khong lay so epoch lam dieu kien: doc that bai thi se
    am tham train lai tu dau, mat ca chuc gio ma khong bao gi.
    """
    last = REPO_DIR / "runs" / name / "weights" / "last.pt"
    if last.exists():
        print("[{}] resume tu epoch {}".format(name, done_epochs(name)))
        YOLO(str(last)).train(resume=True, stop_after_h=STOP_AFTER_H)
    else:
        print("[{}] train tu dau".format(name))
        YOLO(weights).train(data=DATA, epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                            device=DEVICE, seed=0,
                            cos_lr=COS_LR, patience=PATIENCE,
                            warmup_epochs=WARMUP,
                            project=str(REPO_DIR / "runs"), name=name,
                            exist_ok=True, stop_after_h=STOP_AFTER_H, **extra)
    return done_epochs(name)

## 4. Resume

In [ ]:
# Kaggle giet phien o 12h, va phien BI GIET thi khong luu output -> mat sach
# last.pt. Vi the moi phien tu dung o STOP_AFTER_H roi ket thuc binh thuong.
# Lan sau: Add Data -> Your Work -> output lan truoc, cell nay chep runs/ ve.
for name in (BASE_NAME, OURS_NAME):
    dst = REPO_DIR / "runs" / name
    if dst.exists():
        continue
    for src in glob.glob("/kaggle/input/**/runs/" + name, recursive=True):
        if pathlib.Path(src, "weights", "last.pt").exists():
            shutil.copytree(src, dst)
            print("chep ve", name, "<-", src)
            break

# Model da prune cung phai chep ve, neu khong se prune lai (khong sai, chi ton them).
for src in glob.glob("/kaggle/input/**/weights/" + PRUNED.name, recursive=True):
    if not PRUNED.exists():
        PRUNED.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PRUNED)
        print("chep ve", PRUNED.name)
    break

print()
print("baseline:", done_epochs(BASE_NAME), "/", EPOCHS, "epoch")
print("ours    :", done_epochs(OURS_NAME), "/", EPOCHS, "epoch")

## 5. Baseline yolo26m

In [ ]:
n_base = train(BASE_NAME, MODEL)

if n_base < EPOCHS:
    print()
    print("Baseline moi {}/{} epoch - het gio phien nay.".format(n_base, EPOCHS))
    print("Add Data output lan nay roi Save & Run All lai. Cac cell duoi se bo qua.")

## 6. Prune 50%

In [ ]:
# Goi thang cac ham trong pruning/, khong qua script.
from prune_common import load_and_prepare, create_masks, finalize_pruning
from prune_l1norm import compute_l1norm_importance

BEST_BASE = REPO_DIR / "runs" / BASE_NAME / "weights" / "best.pt"

if n_base < EPOCHS:
    print("Bo qua: baseline chua xong.")
elif PRUNED.exists():
    print("Da co", PRUNED.name)
else:
    PRUNED.parent.mkdir(parents=True, exist_ok=True)
    m0, bn_dict, ignore_bn, _chunk, layer_cfg, pruned_yaml = load_and_prepare(
        str(BEST_BASE), str(REPO_DIR / "cfg" / "yolo26m.yaml"), SIZE, None)
    imp = compute_l1norm_importance(m0, bn_dict, ignore_bn)
    masks = create_masks(imp, m0, ignore_bn, layer_cfg, RATIO, 8)
    out = finalize_pruning(m0, masks, pruned_yaml, ignore_bn, str(BEST_BASE),
                           str(REPO_DIR / "weights"), 8, RATIO,
                           method_name="l1norm")
    pathlib.Path(out).replace(PRUNED)
    print("->", PRUNED)

## 7. Finetune + CWD

In [ ]:
if n_base < EPOCHS or not PRUNED.exists():
    print("Bo qua: chua co model da prune.")
    n_ours = 0
else:
    # Teacher la baseline cua chinh size nay.
    n_ours = train(OURS_NAME, str(PRUNED),
                   finetune=True,          # build DetectionModelPruned tu maskbndict
                   kd=True, kd_teacher=str(BEST_BASE), kd_method="cwd",
                   kd_lambda=0.5, kd_layers="neck", kd_warmup=5,
                   cwd_temperature=9.0)
    if n_ours < EPOCHS:
        print()
        print("Ours moi {}/{} epoch - Add Data output lan nay roi chay lai."
              .format(n_ours, EPOCHS))

## 8. Ket qua

In [ ]:
def measure(name):
    best = REPO_DIR / "runs" / name / "weights" / "best.pt"
    if done_epochs(name) < EPOCHS or not best.exists():
        return None
    m = YOLO(str(best))
    r = m.val(data=DATA, imgsz=IMGSZ, batch=BATCH, device=DEVICE.split(",")[0])
    return (sum(p.numel() for p in m.model.parameters()) / 1e6,
            r.box.map50 * 100, r.box.map * 100)

rows = [("YOLO26-" + SIZE.upper(), measure(BASE_NAME)),
        ("Ours-" + SIZE.upper(), measure(OURS_NAME))]

print()
if all(v for _, v in rows):
    print("| Model | Params (M) | AP50 | AP50-95 |")
    print("|---|---:|---:|---:|")
    for label, (par, ap50, ap) in rows:
        print("| {} | {:.2f} | {:.2f} | {:.2f} |".format(label, par, ap50, ap))
    print()
    print("XONG ca hai. Gui lai bang tren.")
else:
    print("baseline:", done_epochs(BASE_NAME), "/", EPOCHS, "epoch")
    print("ours    :", done_epochs(OURS_NAME), "/", EPOCHS, "epoch")
    print()
    print("CHUA XONG - Add Data output lan nay roi Save & Run All lai.")